# Logistic Regression — Kaggle: Breast Cancer Wisconsin

**Dataset:** [Breast Cancer Wisconsin (Diagnostic)](https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data)  
**Task:** Binary classification — predict **Malignant (M)** vs **Benign (B)** from cell nucleus measurements.

**Download first:**
```bash
kaggle datasets download -d uciml/breast-cancer-wisconsin-data \
    -p ../../data/breast_cancer --unzip
```

**Features:** 30 numeric features (radius, texture, perimeter, area, smoothness, ...)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_curve, auc, precision_recall_curve

import sys
sys.path.insert(0, '../..')
from src.utils import set_style, classification_summary, plot_confusion_matrix

set_style()
np.random.seed(42)

## 1. Load & Inspect

In [ ]:
DATA_DIR = Path('../../data/breast_cancer')
csv_files = sorted(DATA_DIR.glob('*.csv'))
print('Files:', [f.name for f in csv_files])

df = pd.read_csv(csv_files[0])
print(f'\nShape: {df.shape}')
df.head()

In [ ]:
print('Columns:', df.columns.tolist())
print('\nDiagnosis value counts:')
print(df['diagnosis'].value_counts())
print('\nNull counts:', df.isnull().sum().sum(), 'total')

## 2. Prepare Data

In [ ]:
# Encode target: M → 1 (Malignant), B → 0 (Benign)
df['label'] = (df['diagnosis'] == 'M').astype(int)

# Drop non-feature columns
drop_cols = ['id', 'diagnosis', 'label']
if 'Unnamed: 32' in df.columns:
    drop_cols.append('Unnamed: 32')

feature_cols = [c for c in df.columns if c not in drop_cols]
print(f'{len(feature_cols)} features')

X = df[feature_cols].values.astype(float)
y = df['label'].values

print(f'X shape: {X.shape}  |  y distribution: {np.bincount(y)}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')
print(f'Train positives: {y_train.sum()} / {len(y_train)}')

## 3. EDA — Top Differentiating Features

In [ ]:
# Mean feature values by class
df_feat = pd.DataFrame(X_train, columns=feature_cols)
df_feat['label'] = y_train

means   = df_feat.groupby('label').mean()
diff    = (means.loc[1] - means.loc[0]).abs().sort_values(ascending=False)
top10   = diff.head(10).index.tolist()

print('Top 10 differentiating features (absolute mean difference):')
print(diff.head(10).round(3))

In [ ]:
# Box plots of top features split by diagnosis
fig, axes = plt.subplots(2, 5, figsize=(17, 7))
axes = axes.flatten()

for i, feat in enumerate(top10):
    data_b = df_feat.loc[df_feat.label == 0, feat]
    data_m = df_feat.loc[df_feat.label == 1, feat]
    axes[i].boxplot([data_b, data_m], labels=['Benign', 'Malignant'],
                    patch_artist=True,
                    boxprops=dict(facecolor='#BFDBFE'),
                    medianprops=dict(color='#2563EB', linewidth=2))
    axes[i].set_title(feat, fontsize=9)

fig.suptitle('Top 10 Features by Diagnostic Difference', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Train Logistic Regression Pipeline

In [ ]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(C=1.0, max_iter=1000, random_state=42)),
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

print('=== Test Set Performance ===')
classification_summary(y_test, y_pred, labels=['Benign', 'Malignant'])

plot_confusion_matrix(y_test, y_pred, labels=['Benign', 'Malignant'],
                      title='Confusion Matrix — Breast Cancer')

## 5. ROC Curve & AUC

In [ ]:
y_proba = pipe.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC
axes[0].plot(fpr, tpr, color='#2563EB', lw=2, label=f'AUC = {roc_auc:.4f}')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1.2, label='Random (AUC=0.5)')
axes[0].fill_between(fpr, tpr, alpha=0.08, color='#2563EB')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()

# Precision-Recall
precision, recall, _ = precision_recall_curve(y_test, y_proba)
pr_auc = auc(recall, precision)
axes[1].plot(recall, precision, color='#DC2626', lw=2, label=f'PR-AUC = {pr_auc:.4f}')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Stratified Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_acc  = cross_val_score(pipe, X, y, cv=cv, scoring='accuracy')
cv_roc  = cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc')
cv_f1   = cross_val_score(pipe, X, y, cv=cv, scoring='f1')

print('5-fold Stratified CV:')
print(f'  Accuracy : {cv_acc.mean():.4f} ± {cv_acc.std():.4f}')
print(f'  ROC-AUC  : {cv_roc.mean():.4f} ± {cv_roc.std():.4f}')
print(f'  F1       : {cv_f1.mean():.4f} ± {cv_f1.std():.4f}')

## 7. Feature Importance

In [ ]:
coefs = pipe.named_steps['model'].coef_[0]
coef_df = pd.DataFrame({'Feature': feature_cols, 'Coefficient': coefs})
coef_df = coef_df.reindex(coef_df['Coefficient'].abs().sort_values(ascending=True).index)
coef_df = coef_df.tail(15)   # top 15

fig, ax = plt.subplots(figsize=(8, 7))
colors = ['#DC2626' if v < 0 else '#2563EB' for v in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Coefficient (positive → more likely Malignant)')
ax.set_title('Top 15 Feature Coefficients — Logistic Regression')
plt.tight_layout()
plt.show()

## Recap

Applied logistic regression to a real medical classification problem:

1. **Data prep** — encoded binary labels, removed non-numeric columns.
2. **EDA** — box plots revealed which measurements most distinguish malignant from benign.
3. **Pipeline** — `StandardScaler` + `LogisticRegression` for leak-free scaling.
4. **Evaluation** — confusion matrix, ROC/PR curves, stratified CV for unbiased estimates.
5. **Feature importance** — large positive coefficients → strongly associated with malignancy.

**You have now completed the core ML mathematics journey:**
- ✅ NumPy  
- ✅ Matplotlib  
- ✅ Linear Regression (math + scratch + sklearn + Kaggle)  
- ✅ Gradient Descent (batch / mini-batch / SGD)  
- ✅ Logistic Regression (math + scratch + sklearn + Kaggle)